# PDCH Text Modality — Preprocessing Pipeline (v1)

## 0. Config & imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Drive mounted")

In [ ]:
import os, re, glob, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path('/content/drive/MyDrive/Depression_FL_Project/data/raw/PDCH')
WAV_DIR    = DATA_ROOT / 'wav_data'
LABEL_XLSX = DATA_ROOT / 'HAMD_annotation_en.xlsx'  # English item/total headers
OUT_DIR = Path('/content/drive/MyDrive/Depression_FL_Project/output/PDCH/processed_v1')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR    = OUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

In [ ]:
import glob, os
base = '/content/drive/MyDrive/Depression_FL_Project/data/raw/PDCH'
print("Does base exist?", os.path.exists(base), "\n")
print("Contents of", base, ":")
for p in sorted(glob.glob(base + '/*')):
    print("   ", p, "(dir)" if os.path.isdir(p) else "(file)")

print("\nSearching for wav_data under that base (any depth):")
print("  ", glob.glob(base + '/**/wav_data', recursive=True))

print("\nSearching for any patient folder 001A under that base:")
print("  ", glob.glob(base + '/**/001A', recursive=True)[:3])

## 1. HAMD-17 item names & scoring reference

In [ ]:
HAMD17_ITEMS = {
    1:  'depressed_mood',
    2:  'guilt',
    3:  'suicide',
    4:  'insomnia_early',
    5:  'insomnia_middle',
    6:  'insomnia_late',
    7:  'work_activities',
    8:  'retardation',
    9:  'agitation',
    10: 'anxiety_psychic',
    11: 'anxiety_somatic',
    12: 'somatic_gastro',
    13: 'somatic_general',
    14: 'genital_symptoms',
    15: 'hypochondriasis',
    16: 'weight_loss',
    17: 'insight',
}
MISSING_CODE = 9   # 'uncertain / inapplicable' per the data descriptor

SEVERITY_BANDS_5 = [(-1, 7, 0, 'normal'), (7, 13, 1, 'mild'),
                    (13, 18, 2, 'moderate'), (18, 22, 3, 'severe'),
                    (22, 999, 4, 'very_severe')]
SEVERITY_BANDS_3 = [(-1, 7, 0, 'none_minimal'), (7, 17, 1, 'mild_moderate'),
                    (17, 999, 2, 'severe')]

def to_band(total, bands):
    if pd.isna(total): return (np.nan, None)
    for lo, hi, code, name in bands:
        if lo < total <= hi: return (code, name)
    return (np.nan, None)

## 2. Transcript parser

In [ ]:
ROLE_MAP = {'患者': 'patient', '医生': 'doctor'}
TURN_RE  = re.compile(r'^\s*(患者|医生)\s*[：:]\s*(.*)$')

def _seg_index(path):
    m = re.match(r'^(\d+)_correction\.txt$', os.path.basename(path))
    return int(m.group(1)) if m else 0

def parse_correction_file(path):
    """Return list of (role, text) for non-empty turns in one _correction.txt segment."""
    out = []
    for raw in Path(path).read_text(encoding='utf-8').splitlines():
        m = TURN_RE.match(raw)
        if not m:
            continue
        role, text = ROLE_MAP[m.group(1)], m.group(2).strip()
        if text:               # skip blank-content turns
            out.append((role, text))
    return out

def load_patient_turns(patient_dir):
    """All turns for one patient folder, segments concatenated in numeric order."""
    seg_files = sorted(glob.glob(os.path.join(patient_dir, '*_correction.txt')), key=_seg_index)
    rows, turn_idx = [], 0
    for sf in seg_files:
        seg = _seg_index(sf)
        for role, text in parse_correction_file(sf):
            rows.append(dict(segment=seg, turn_idx=turn_idx, role=role,
                             text=text, n_chars=len(text)))
            turn_idx += 1
    return rows, len(seg_files)

In [ ]:
import glob
hits = glob.glob('/content/drive/MyDrive/**/wav_data', recursive=True)
print("Found wav_data at:")
for h in hits:
    print("   ", h)

In [ ]:
import glob, os

print("Drive mounted?", os.path.exists('/content/drive/MyDrive'), "\n")

print("wav_data anywhere in Drive:")
print("  ", glob.glob('/content/drive/**/wav_data', recursive=True))

print("\nwav_data anywhere on ephemeral /content (non-Drive):")
print("  ", [h for h in glob.glob('/content/**/wav_data', recursive=True) if '/drive/' not in h])

print("\nAny patient folders (NNN A/B) anywhere:")
print("  ", glob.glob('/content/**/001A', recursive=True)[:3])

print("\nAny PDCH zip still around:")
print("  ", glob.glob('/content/**/*.zip', recursive=True) + glob.glob('/content/drive/**/*PDCH*', recursive=True)[:10])

## 3. Build the turn-level table

In [ ]:
assert WAV_DIR.exists(), f'wav_data not found at {WAV_DIR} — fix DATA_ROOT in cell 0'
patient_dirs = sorted([p for p in WAV_DIR.iterdir() if p.is_dir()])
print(f'Found {len(patient_dirs)} patient folders')

turn_rows, seg_counts = [], {}
for pdir in patient_dirs:
    pid = pdir.name
    rows, n_seg = load_patient_turns(str(pdir))
    seg_counts[pid] = n_seg
    for r in rows:
        r['patient_id'] = pid
        turn_rows.append(r)
    if not rows:
        print(f'  ⚠ no turns parsed for {pid}')

turns = pd.DataFrame(turn_rows)[['patient_id','segment','turn_idx','role','text','n_chars']]
print('Turn table:', turns.shape)
print('Patients with turns:', turns.patient_id.nunique())
print('Avg turns/patient:', round(turns.groupby('patient_id').size().mean(), 1))
turns.head()

## 4. Load & clean HAMD labels

In [ ]:
raw = pd.read_excel(LABEL_XLSX, sheet_name='Sheet1', dtype={'Serial': str})
id_col = 'Serial' if 'Serial' in raw.columns else raw.columns[0]
item_cols = [c for c in raw.columns if isinstance(c, (int, np.integer))]
assert len(item_cols) == 17, f'expected 17 item cols, got {len(item_cols)}'

lab = raw[[id_col] + item_cols].rename(columns={id_col: 'patient_id'}).copy()
lab = lab.rename(columns={i: f'hamd_{i:02d}_{HAMD17_ITEMS[i]}' for i in item_cols})
item_named = [f'hamd_{i:02d}_{HAMD17_ITEMS[i]}' for i in item_cols]

# 9 -> NaN (uncertain/inapplicable)
n9 = int((lab[item_named] == MISSING_CODE).sum().sum())
lab[item_named] = lab[item_named].replace(MISSING_CODE, np.nan)
print(f'Replaced {n9} score-9 cells with NaN (uncertain/inapplicable).')

# recompute total (NaN-aware)
lab['hamd_total'] = lab[item_named].sum(axis=1, skipna=True)
lab['n_missing_items'] = lab[item_named].isna().sum(axis=1)

# severity bins
lab[['sev5_code','sev5_name']] = lab['hamd_total'].apply(lambda t: pd.Series(to_band(t, SEVERITY_BANDS_5)))
lab[['sev3_code','sev3_name']] = lab['hamd_total'].apply(lambda t: pd.Series(to_band(t, SEVERITY_BANDS_3)))

print('Label rows:', len(lab), '| unique IDs:', lab.patient_id.nunique())
print('\n5-class severity (our standard bands):')
print(lab['sev5_name'].value_counts())
print('\n3-class severity:')
print(lab['sev3_name'].value_counts())
lab.head()

## 5. Aggregate transcripts to patient level + join labels

In [ ]:
def agg_patient(g):
    pt = g[g.role == 'patient']
    dr = g[g.role == 'doctor']
    p_chars, d_chars = int(pt.n_chars.sum()), int(dr.n_chars.sum())
    return pd.Series({
        'n_turns': len(g),
        'n_patient_turns': len(pt),
        'n_doctor_turns': len(dr),
        'patient_chars': p_chars,
        'doctor_chars': d_chars,
        'talk_ratio': p_chars / max(1, d_chars),   # patient/doctor (paper's Small/Large axis)
        'patient_text': '\n'.join(pt.text.tolist()),
        'full_dialogue': '\n'.join(f'[{r}] {t}' for r, t in zip(g.role, g.text)),
    })

pat = turns.groupby('patient_id').apply(agg_patient, include_groups=False).reset_index()

med = pat['talk_ratio'].median()
pat['talk_bucket'] = np.where(pat['talk_ratio'] >= med, 'Large', 'Small')

df = pat.merge(lab, on='patient_id', how='outer', indicator=True)
print('Join breakdown:')
print(df['_merge'].value_counts())
only_txt = df[df._merge=='left_only'].patient_id.tolist()
only_lab = df[df._merge=='right_only'].patient_id.tolist()
if only_txt: print('  transcript but NO label (drop):', only_txt)
if only_lab: print('  label but NO transcript (drop):', only_lab)

data = df[df._merge=='both'].drop(columns='_merge').reset_index(drop=True)
print('\nFinal joined patients:', len(data))

## 6. Integrity & leakage checks

In [ ]:
checks = {}
checks['unique_patient_ids']      = data.patient_id.is_unique
checks['no_empty_patient_text']   = (data.patient_text.str.len() > 0).all()
checks['all_items_in_0_4_or_nan'] = data[item_named].apply(
    lambda c: c.dropna().between(0,4).all()).all()
checks['total_matches_itemsum']   = np.allclose(
    data['hamd_total'], data[item_named].sum(axis=1, skipna=True))
checks['no_score9_remaining']     = int((data[item_named]==9).sum().sum()) == 0
checks['one_row_per_patient']     = (data.groupby('patient_id').size()==1).all()

for k,v in checks.items():
    print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(checks.values()), 'Integrity check failed — inspect above.'
print('\nAll integrity checks passed.')

## 7. EDA — replicate the paper's Figs 4–6 as a sanity check

In [ ]:
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':300, 'font.size':10})

# Fig 5 analogue: per-turn avg char length, patient vs doctor
fig, ax = plt.subplots(figsize=(7,4))
p_avg = turns[turns.role=='patient'].groupby('patient_id').n_chars.mean()
d_avg = turns[turns.role=='doctor'].groupby('patient_id').n_chars.mean()
bins = range(0, 81, 10)
ax.hist(d_avg, bins=bins, alpha=.7, label='Doctor')
ax.hist(p_avg, bins=bins, alpha=.7, label='Patient')
ax.set_xlabel('Avg characters per turn'); ax.set_ylabel('# patients')
ax.set_title('Per-turn utterance length (cf. paper Fig 5)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'fig5_utterance_length.png'); plt.show()

In [ ]:
# Fig 6 analogue: per-item HAMD score distribution (stacked)
counts = []
for c in item_named:
    vc = data[c].value_counts(dropna=False)
    counts.append([vc.get(s,0) for s in [0,1,2,3,4]] + [data[c].isna().sum()])
cdf = pd.DataFrame(counts, index=[c.split('_',2)[2] for c in item_named],
                   columns=['0','1','2','3','4','N/A'])
ax = cdf.plot(kind='bar', stacked=True, figsize=(12,5), width=.8)
ax.set_ylabel('# patients'); ax.set_title('HAMD-17 per-item score distribution (cf. paper Fig 6)')
ax.legend(title='score', bbox_to_anchor=(1.01,1)); plt.tight_layout()
plt.savefig(FIG_DIR/'fig6_hamd_item_dist.png'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14,4))
axes[0].hist(data.talk_ratio, bins=20); axes[0].axvline(med, color='r', ls='--', label=f'median={med:.2f}')
axes[0].set_title('Patient/Doctor talk ratio'); axes[0].set_xlabel('ratio'); axes[0].legend()
data.sev5_name.value_counts().reindex([b[3] for b in SEVERITY_BANDS_5]).plot(kind='bar', ax=axes[1])
axes[1].set_title('5-class severity'); axes[1].tick_params(axis='x', rotation=45)
data.sev3_name.value_counts().reindex([b[3] for b in SEVERITY_BANDS_3]).plot(kind='bar', ax=axes[2], color='C2')
axes[2].set_title('3-class severity'); axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig(FIG_DIR/'fig_severity_talkratio.png'); plt.show()

## 8. Save processed artifacts

In [ ]:
turns.to_parquet(OUT_DIR/'pdch_turns.parquet', index=False)

patient_cols = ['patient_id','n_turns','n_patient_turns','n_doctor_turns',
                'patient_chars','doctor_chars','talk_ratio','talk_bucket',
                'hamd_total','n_missing_items',
                'sev5_code','sev5_name','sev3_code','sev3_name',
                *item_named, 'patient_text','full_dialogue']
data[patient_cols].to_parquet(OUT_DIR/'pdch_patients.parquet', index=False)

manifest = data[['patient_id','talk_ratio','talk_bucket','hamd_total',
                 'sev5_code','sev5_name','sev3_code','sev3_name','n_missing_items']]
manifest.to_csv(OUT_DIR/'pdch_manifest.csv', index=False)

summary = {
    'n_patients': int(len(data)),
    'n_turns': int(len(turns)),
    'dropped_transcript_no_label': only_txt,
    'dropped_label_no_transcript': only_lab,
    'score9_cells_set_nan': n9,
    'sev5_distribution': data.sev5_name.value_counts().to_dict(),
    'sev3_distribution': data.sev3_name.value_counts().to_dict(),
    'talk_ratio_median': float(med),
}
json.dump(summary, open(OUT_DIR/'pdch_summary.json','w'), ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nSaved to', OUT_DIR.resolve())

## Next steps (FL, later)